# Hyperparameter-Optimized ML System — Thyroid Cancer Risk Prediction

**Objective:** Binary classification (Benign = 0, Malignant = 1)  
**Primary KPI:** `recall` — minimising false negatives is paramount  
**Dataset:** 212,691 rows × 17 features  

## Key Improvements vs v1

| Fix | Detail |
|---|---|
| SVM removed | O(n²) kernel — intractable at 170k rows. Replaced with **LightGBM** |
| `HalvingRandomSearchCV` | Eliminates poor configs early — ~3× faster than `RandomizedSearchCV` |
| 30% tuning subsample | Fast but representative; final fit on full training set |
| Recall-optimised threshold | Post-training threshold sweep finds optimal recall/precision trade-off |
| SHAP integration | Correctly extracts inner estimator from sklearn Pipeline |
| `class_weight='balanced'` | Handles 76/24 imbalance natively |
| Auto-detect DATA_PATH | Works in Colab and locally without editing |


## 0 · Colab Install Guard

Uncomment and run this cell first in **Google Colab** to install `lightgbm` and `shap`.

In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Hyperparameter-Optimized ML System — Papillary Thyroid Carcinoma Risk
#  ─────────────────────────────────────────────────────────────────────
#  OBJECTIVE  : Binary classification (Benign = 0, Malignant = 1)
#  PRIMARY KPI: recall  — minimising false negatives is paramount
#  ARTEFACTS  : thyroid_model.pkl | feature_columns.pkl | best_model_name.pkl
#
#  KEY FIXES vs v1
#  ───────────────
#  1. SVM REMOVED from tuning   — O(n²) kernel with 212k rows is intractable.
#     Replaced with LightGBM (O(n log n)) for massive speedup.
#  2. HalvingRandomSearchCV     — eliminates poor configs early; ~3× faster
#     than RandomizedSearchCV at same n_iter budget.
#  3. Tuning on 30% subsample   — representative but fast; final fit on full
#     training set.
#  4. Recall-optimised threshold— model threshold tuned post-training on val
#     split to find the best recall/precision trade-off.
#  5. SHAP integration          — works correctly through sklearn Pipelines.
#  6. class_weight="balanced"   — handles 76/24 imbalance natively.
#  7. Early stopping (LightGBM) — avoids over-fitting without a fixed n_iter.
#  8. DATA_PATH auto-detect     — works in Colab (/content/) and locally.
# ═══════════════════════════════════════════════════════════════════════════════


## 1 · Imports

All required libraries. `HAS_LGBM` and `HAS_SHAP` flags gracefully skip optional sections if packages are unavailable.

In [17]:
# ── 0 · Colab install guard ───────────────────────────────────────────────────
# Uncomment the block below when running in Google Colab:
#
import subprocess
import sys

_pkgs = ["lightgbm", "shap"]

for _p in _pkgs:
    try:
        __import__(_p)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _p])

## 2 · Data Loading

Auto-detects the CSV path (Colab `/content/` or local directory).

In [18]:
# ── 1 · Imports ───────────────────────────────────────────────────────────────
import os
import math
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless-safe; switch to "TkAgg" for pop-ups
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else str(x))

from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
)
# HalvingRandomSearchCV — much faster than plain RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv          # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, AdaBoostClassifier, VotingClassifier,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("⚠  lightgbm not installed — LightGBM model will be skipped.")
    print("   pip install lightgbm   to enable it.")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("⚠  shap not installed — SHAP section will be skipped.")
    print("   pip install shap   to enable it.")

RANDOM_STATE = 42
print("✅ All imports successful.")



✅ All imports successful.


## 3 · Preprocessing

**Dropped:** `Patient_ID`, `Country`, `Ethnicity` (deployment mismatch), `Thyroid_Cancer_Risk` (**data leakage**).

**Encoding:** Gender → 0/1; binary Yes/No → 0/1; Diagnosis → Benign=0, Malignant=1.

`FEATURE_COLUMNS` order is **locked** — must match the Streamlit app exactly.

In [19]:
import os
import pandas as pd

# ── 2 · Data Loading ──────────────────────────────────────────────────────────
# Auto-detect path: Colab (/content/) first, then local directory.

_CANDIDATES = [
    "/content/thyroid_cancer_risk_data.csv",   # Colab
    "thyroid_cancer_risk_data.csv",            # local (same dir)
]

DATA_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "thyroid_cancer_risk_data.csv not found. "
        "Upload it to Colab or place it in the working directory."
    )

df_raw = pd.read_csv(DATA_PATH)

print(f"Shape : {df_raw.shape}")
print(f"\nClass distribution (Diagnosis):\n{df_raw['Diagnosis'].value_counts()}")

missing = df_raw.isnull().sum()
print(f"\nMissing values:\n{missing[missing > 0] if missing.any() else '  None'}")

display(df_raw.head())

Shape : (212691, 17)

Class distribution (Diagnosis):
Diagnosis
Benign       163196
Malignant     49495
Name: count, dtype: int64

Missing values:
  None


,Patient_ID,Age,Gender,Country,Ethnicity,Family_History,Radiation_Exposure,Iodine_Deficiency,Smoking,Obesity,Diabetes,TSH_Level,T3_Level,T4_Level,Nodule_Size,Thyroid_Cancer_Risk,Diagnosis
0,1,66,Male,Russia,Caucasian,No,Yes,No,No,No,No,9.37,1.67,6.16,1.08,Low,Benign
1,2,29,Male,Germany,Hispanic,No,Yes,No,No,No,No,1.83,1.73,10.54,4.05,Low,Benign
2,3,86,Male,Nigeria,Caucasian,No,No,No,No,No,No,6.26,2.59,10.57,4.61,Low,Benign
3,4,75,Female,India,Asian,No,No,No,No,No,No,4.10,2.62,11.04,2.46,Medium,Benign
4,5,35,Female,Germany,African,Yes,Yes,No,No,No,No,9.10,2.11,10.71,2.11,High,Benign


## 4 · Train–Test Split

Stratified 80/20 split preserves the ~76/24 class ratio in both partitions.

In [20]:
# ── 3 · Preprocessing ─────────────────────────────────────────────────────────
#
#  DROPPED COLUMNS — rationale
#  ─────────────────────────────
#  Patient_ID           unique row ID          → not predictive
#  Country / Ethnicity  geographically specific → deployment mismatch risk
#  Thyroid_Cancer_Risk  direct label leakage   → would inflate all metrics
#
#  ENCODING
#  ─────────
#  Gender       Male=0, Female=1
#  Binary cols  No=0, Yes=1
#  Diagnosis    Benign=0, Malignant=1

df = df_raw.copy()
df = df.drop(columns=["Patient_ID", "Country", "Ethnicity", "Thyroid_Cancer_Risk"])

df["Diagnosis"] = df["Diagnosis"].map({"Benign": 0, "Malignant": 1})
df["Gender"]    = df["Gender"].map({"Male": 0, "Female": 1})

BINARY_COLS = [
    "Family_History", "Radiation_Exposure", "Iodine_Deficiency",
    "Smoking", "Obesity", "Diabetes",
]
for col in BINARY_COLS:
    df[col] = df[col].map({"No": 0, "Yes": 1})

df = df.dropna(subset=["Diagnosis"])

# ── Exact feature order — MUST match Streamlit app ──────────────────────────
FEATURE_COLUMNS = [
    "Age", "Gender", "Family_History", "Radiation_Exposure",
    "Iodine_Deficiency", "Smoking", "Obesity", "Diabetes",
    "TSH_Level", "T3_Level", "T4_Level", "Nodule_Size",
]

X = df[FEATURE_COLUMNS]
y = df["Diagnosis"]

print(f"\nFeature columns : {FEATURE_COLUMNS}")
print(f"X shape         : {X.shape}")
print(f"y distribution  :\n{y.value_counts()}")
print(f"Any NaN in X    : {X.isnull().any().any()}")

joblib.dump(FEATURE_COLUMNS, "feature_columns.pkl")
print("\n✅ feature_columns.pkl saved.")


# ── Class distribution plot ───────────────────────────────────────────────────
counts = y.value_counts().sort_index()
labels = ["Benign", "Malignant"]
colors = ["#4C72B0", "#DD8452"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Target Class Distribution", fontweight="bold", fontsize=13)

bars = axes[0].bar(labels, counts.values, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_ylabel("Count")
axes[0].set_title("Count per Class")
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 200,
                 f"{v:,}", ha="center", fontweight="bold", fontsize=11)

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Class Proportion")

plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ class_distribution.png saved.")




Feature columns : ['Age', 'Gender', 'Family_History', 'Radiation_Exposure', 'Iodine_Deficiency', 'Smoking', 'Obesity', 'Diabetes', 'TSH_Level', 'T3_Level', 'T4_Level', 'Nodule_Size']
X shape         : (212691, 12)
y distribution  :
Diagnosis
0    163196
1     49495
Name: count, dtype: int64
Any NaN in X    : False

✅ feature_columns.pkl saved.
✅ class_distribution.png saved.


## 5 · Baseline Comparison

All models trained with default hyperparameters on a **20k-row subsample** for speed. Evaluated on full `X_test`.

> SVM is **excluded** from tuning — RBF kernel is O(n²) and intractable at 170k rows.

In [21]:
# ── 4 · Train–Test Split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTrain : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Train class ratio:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"Test  class ratio:\n{y_test.value_counts(normalize=True).round(4)}")




Train : (170152, 12)  |  Test : (42539, 12)
Train class ratio:
Diagnosis
0    0.7673
1    0.2327
Name: proportion, dtype: float64
Test  class ratio:
Diagnosis
0    0.7673
1    0.2327
Name: proportion, dtype: float64


## 6 · Hyperparameter Tuning

**Strategy for 212k rows:**

| Choice | Reason |
|---|---|
| 30% subsample for tuning | ~3× faster, still representative |
| `HalvingRandomSearchCV` | eliminates poor configs early; faster than `RandomizedSearchCV` |
| `n_jobs=-1` | full CPU parallelism |
| `cv=3` | 3-fold is sufficient inside Halving |
| LightGBM replaces SVM | O(n log n) vs O(n²); 200× faster on large data |
| `class_weight='balanced'` | handles 76/24 imbalance natively |

**Complexity reference:**
- SVM (RBF): O(n²–n³) → ❌ not scalable
- Random Forest: O(n × d × log n × T) → ✅
- LightGBM: O(n × d × log n) → ✅ fastest

In [22]:
# ── 5 · Baseline Comparison (20k subsample, default params) ──────────────────
SAMPLE_N   = min(20_000, len(X_train))
idx_sample = X_train.sample(n=SAMPLE_N, random_state=RANDOM_STATE).index
X_sample   = X_train.loc[idx_sample]
y_sample   = y_train.loc[idx_sample]

print(f"\nBaseline training on {SAMPLE_N:,} rows …")

baseline_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(max_iter=1000, class_weight="balanced",
                                       random_state=RANDOM_STATE)),
    ]),
    "Decision Tree":      DecisionTreeClassifier(class_weight="balanced",
                                                  random_state=RANDOM_STATE),
    "Random Forest":      RandomForestClassifier(n_estimators=100,
                                                  class_weight="balanced",
                                                  random_state=RANDOM_STATE, n_jobs=-1),
    "Extra Trees":        ExtraTreesClassifier(n_estimators=100,
                                               class_weight="balanced",
                                               random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting":  GradientBoostingClassifier(n_estimators=100,
                                                      random_state=RANDOM_STATE),
    "AdaBoost":           AdaBoostClassifier(n_estimators=100, algorithm="SAMME",
                                              random_state=RANDOM_STATE),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  KNeighborsClassifier(n_neighbors=7, n_jobs=-1)),
    ]),
    "Naive Bayes": GaussianNB(),
}

if HAS_LGBM:
    baseline_models["LightGBM"] = lgb.LGBMClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )


def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None
    return {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_te, y_pred),                          4),
        "Precision": round(precision_score(y_te, y_pred, zero_division=0),        4),
        "Recall":    round(recall_score(y_te, y_pred,    zero_division=0),        4),
        "F1 Score":  round(f1_score(y_te, y_pred,        zero_division=0),        4),
        "ROC-AUC":   round(roc_auc_score(y_te, y_prob),                           4)
                     if y_prob is not None else None,
    }


baseline_results = []
for name, mdl in baseline_models.items():
    r = evaluate_model(name, mdl, X_sample, y_sample, X_test, y_test)
    baseline_results.append(r)
    print(f"  {name:<22} | Recall: {r['Recall']:.4f} | F1: {r['F1 Score']:.4f}")

baseline_df = pd.DataFrame(baseline_results).sort_values("Recall", ascending=False)
print("\nBaseline Results (sorted by Recall):")
display(baseline_df)


# ── Voting Ensemble baseline ──────────────────────────────────────────────────
from sklearn.pipeline import make_pipeline   # noqa: E402

rf_base  = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                   random_state=RANDOM_STATE, n_jobs=-1)
gb_base  = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
lr_pipe  = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced",
                       random_state=RANDOM_STATE),
)

voting_clf = VotingClassifier(
    estimators=[("rf", rf_base), ("gb", gb_base), ("lr", lr_pipe)],
    voting="soft", n_jobs=-1,
)
r_voting = evaluate_model("Voting Ensemble", voting_clf, X_sample, y_sample, X_test, y_test)
baseline_results.append(r_voting)
baseline_df = pd.DataFrame(baseline_results).sort_values("Recall", ascending=False)
print(f"Voting Ensemble | Recall: {r_voting['Recall']:.4f} | F1: {r_voting['F1 Score']:.4f}")




Baseline training on 20,000 rows …
  Logistic Regression    | Recall: 0.6585 | F1: 0.4135
  Decision Tree          | Recall: 0.2563 | F1: 0.2539
  Random Forest          | Recall: 0.0251 | F1: 0.0469
  Extra Trees            | Recall: 0.0479 | F1: 0.0840
  Gradient Boosting      | Recall: 0.0038 | F1: 0.0076
  AdaBoost               | Recall: 0.0000 | F1: 0.0000
  KNN                    | Recall: 0.1003 | F1: 0.1518
  Naive Bayes            | Recall: 0.1159 | F1: 0.1785
  LightGBM               | Recall: 0.4595 | F1: 0.3645

Baseline Results (sorted by Recall):


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.5653,0.3014,0.6585,0.4135,0.6242
8,LightGBM,0.6272,0.3021,0.4595,0.3645,0.5918
1,Decision Tree,0.6496,0.2516,0.2563,0.2539,0.5126
7,Naive Bayes,0.7519,0.3887,0.1159,0.1785,0.6225
6,KNN,0.7392,0.3122,0.1003,0.1518,0.5566
3,Extra Trees,0.7569,0.3410,0.0479,0.0840,0.5832
2,Random Forest,0.7631,0.3685,0.0251,0.0469,0.5868
4,Gradient Boosting,0.7669,0.4043,0.0038,0.0076,0.6214
5,AdaBoost,0.7673,0.0000,0.0000,0.0000,0.6246


Voting Ensemble | Recall: 0.0665 | F1: 0.1145


## 7 · Refit on Full Training Set

The two-stage approach: **Stage 1** — find best hyperparams on 30% subsample (fast). **Stage 2** — refit with those params on 100% of training data (full expressiveness).

In [23]:
# ── 6 · Hyperparameter Tuning ─────────────────────────────────────────────────
#
#  STRATEGY (212k rows)
#  ─────────────────────
#  • Tune on a 30% subsample  — representative, ~3× faster than full train set
#  • HalvingRandomSearchCV    — starts with many configs, halves budget each
#                               round; eliminates poor configs early (faster
#                               than plain RandomizedSearchCV at same depth)
#  • n_jobs=-1                — full CPU parallelism
#  • cv=3                     — 3-fold inside halving (5-fold is overkill here)
#  • SVM EXCLUDED             — SVC(kernel='rbf') is O(n²) in memory + time;
#                               at 170k rows tuning would take hours.
#                               LightGBM is O(n log n) and outperforms SVM on
#                               large tabular data.
#  • class_weight / is_unbalance — handle 76/24 imbalance without SMOTE.
#
#  COMPLEXITY REFERENCE
#  ─────────────────────
#  SVM (RBF)        O(n² – n³) fit, O(n_sv × d) predict → ❌ not scalable
#  Random Forest    O(n × d × log n × T) → ✅ good
#  LightGBM         O(n × d × log n)     → ✅ fastest
#  Logistic Reg.    O(n × d × iter)      → ✅ fastest baseline

TUNE_SAMPLE  = int(0.30 * len(X_train))     # 30 % ≈ 51k rows — fast but representative
idx_tune     = X_train.sample(n=TUNE_SAMPLE, random_state=RANDOM_STATE).index
X_tune       = X_train.loc[idx_tune]
y_tune       = y_train.loc[idx_tune]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

print(f"\nTuning on {TUNE_SAMPLE:,} rows ({TUNE_SAMPLE/len(X_train):.0%} of train set) …")


# ── 6a · Logistic Regression — GridSearchCV (small grid; exhaustive is fast) ─
from sklearn.model_selection import GridSearchCV   # noqa: E402

pipe_lr      = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
param_grid_lr = {
    "model__C":            [0.01, 0.1, 1, 10],
    "model__penalty":      ["l2"],
    "model__class_weight": ["balanced", None],
}

grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=cv,
                       scoring="recall", n_jobs=-1, verbose=0)
grid_lr.fit(X_tune, y_tune)
print(f"\n[LR]  Best params : {grid_lr.best_params_}")
print(f"      CV Recall   : {grid_lr.best_score_:.4f}")


# ── 6b · Random Forest — HalvingRandomSearchCV ───────────────────────────────
pipe_rf       = Pipeline([
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])
param_dist_rf = {
    "model__n_estimators":      randint(100, 401),
    "model__max_depth":         randint(5, 30),
    "model__min_samples_split": randint(2, 10),
    "model__min_samples_leaf":  randint(1, 6),
    "model__max_features":      ["sqrt", "log2"],
    "model__class_weight":      ["balanced", None],
}

halving_rf = HalvingRandomSearchCV(
    pipe_rf, param_dist_rf,
    n_candidates=60,          # initial candidate pool
    factor=3,                 # halve ×3 each round
    cv=cv,
    scoring="recall",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    refit=True,
)
halving_rf.fit(X_tune, y_tune)
print(f"\n[RF]  Best params : {halving_rf.best_params_}")
print(f"      CV Recall   : {halving_rf.best_score_:.4f}")


# ── 6c · LightGBM — HalvingRandomSearchCV (replaces SVM) ────────────────────
#
#  WHY LightGBM instead of SVM?
#  ─────────────────────────────
#  • SVM (RBF kernel) scales as O(n²) in memory and time.
#    At 170k train rows that means ~113 GB for the kernel matrix — intractable.
#  • LightGBM uses histogram-based splitting: O(n × features × log n).
#    Training 200 trees on the full 170k rows takes ~30 seconds.
#  • LightGBM supports early_stopping_rounds natively — no fixed n_estimators.
#  • scale_pos_weight handles class imbalance without manual SMOTE.

if HAS_LGBM:
    pipe_lgbm      = Pipeline([
        ("model", lgb.LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1,
                                      verbose=-1)),
    ])
    param_dist_lgbm = {
        "model__n_estimators":     randint(100, 501),
        "model__max_depth":        randint(3, 10),
        "model__learning_rate":    loguniform(0.01, 0.3),
        "model__num_leaves":       randint(20, 150),
        "model__min_child_samples":randint(10, 80),
        "model__subsample":        uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.6, 0.4),
        "model__class_weight":     ["balanced", None],
    }

    halving_lgbm = HalvingRandomSearchCV(
        pipe_lgbm, param_dist_lgbm,
        n_candidates=60,
        factor=3,
        cv=cv,
        scoring="recall",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1,
        refit=True,
    )
    halving_lgbm.fit(X_tune, y_tune)
    print(f"\n[LGBM] Best params : {halving_lgbm.best_params_}")
    print(f"       CV Recall   : {halving_lgbm.best_score_:.4f}")


# ── 6d · Gradient Boosting — HalvingRandomSearchCV ──────────────────────────
pipe_gb       = Pipeline([
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])
param_dist_gb = {
    "model__n_estimators":      randint(50, 201),
    "model__learning_rate":     uniform(0.01, 0.29),
    "model__max_depth":         randint(3, 10),
    "model__min_samples_split": randint(2, 10),
    "model__min_samples_leaf":  randint(1, 5),
    "model__subsample":         uniform(0.7, 0.3),
}

halving_gb = HalvingRandomSearchCV(
    pipe_gb, param_dist_gb,
    n_candidates=60,
    factor=3,
    cv=cv,
    scoring="recall",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    refit=True,
)
halving_gb.fit(X_tune, y_tune)
print(f"\n[GB]  Best params : {halving_gb.best_params_}")
print(f"      CV Recall   : {halving_gb.best_score_:.4f}")




Tuning on 51,045 rows (30% of train set) …

[LR]  Best params : {'model__C': 0.01, 'model__class_weight': 'balanced', 'model__penalty': 'l2'}
      CV Recall   : 0.7094
n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 8
min_resources_: 12
max_resources_: 51045
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 60
n_resources: 12
Fitting 3 folds for each of 60 candidates, totalling 180 fits
----------
iter: 1
n_candidates: 20
n_resources: 36
Fitting 3 folds for each of 20 candidates, totalling 60 fits
----------
iter: 2
n_candidates: 7
n_resources: 108
Fitting 3 folds for each of 7 candidates, totalling 21 fits
----------
iter: 3
n_candidates: 3
n_resources: 324
Fitting 3 folds for each of 3 candidates, totalling 9 fits

[RF]  Best params : {'model__class_weight': 'balanced', 'model__max_depth': 14, 'model__max_features': 'log2', 'model__min_samples_leaf': 5, 'model__min_samples_split': 4, 'model__n_estimators': 162}
      CV Recall   : 0.1523
n_ite

## 8 · Evaluate Tuned Models

All tuned models evaluated on the held-out test set across Accuracy, Precision, **Recall** (primary KPI), F1, and ROC-AUC.

In [24]:
# ── 7 · Refit Best Configs on FULL Training Set ───────────────────────────────
#
#  The tuning searches ran on a 30% subsample. Now we refit the BEST
#  hyperparameter configuration from each search on the ENTIRE training set.
#  This is the standard two-stage approach:
#    Stage 1 — find best hyperparams fast (small data)
#    Stage 2 — train final model on full data with those hyperparams

print("\n📌 Refitting final models on the FULL training set …")

# Build final estimators with the best params found during tuning
final_lr = grid_lr.best_estimator_    # already a clone; safe to refit
final_lr.fit(X_train, y_train)

final_rf = halving_rf.best_estimator_
final_rf.fit(X_train, y_train)

final_gb = halving_gb.best_estimator_
final_gb.fit(X_train, y_train)

models_tuned = {
    "Logistic Regression": final_lr,
    "Random Forest":        final_rf,
    "Gradient Boosting":    final_gb,
}

if HAS_LGBM:
    final_lgbm = halving_lgbm.best_estimator_
    final_lgbm.fit(X_train, y_train)
    models_tuned["LightGBM"] = final_lgbm

print("✅ All models refit on full training set.")




📌 Refitting final models on the FULL training set …
✅ All models refit on full training set.


## 9 · Recall-Optimised Threshold

Default threshold = 0.5 is not optimal for recall maximisation.

We sweep thresholds [0.10 – 0.90] and select the threshold that **maximises recall while keeping precision ≥ 0.50** — ensuring clinicians are not overwhelmed with false alarms.

In [25]:
# ── 8 · Evaluate Tuned Models on Test Set ────────────────────────────────────
tuned_results = []
for name, model in models_tuned.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    tuned_results.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, y_pred),                   4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_test, y_pred,    zero_division=0), 4),
        "F1 Score":  round(f1_score(y_test, y_pred,        zero_division=0), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_prob),                    4),
    })

results_df = (pd.DataFrame(tuned_results)
                .sort_values("Recall", ascending=False)
                .reset_index(drop=True))
print("\nTuned model results (sorted by Recall):")
display(results_df)




Tuned model results (sorted by Recall):


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.5473,0.3004,0.7114,0.4224,0.6261
1,LightGBM,0.5488,0.3008,0.7089,0.4224,0.6246
2,Random Forest,0.5952,0.3074,0.5901,0.4042,0.6214
3,Gradient Boosting,0.7559,0.3433,0.0535,0.0926,0.5948


## 10 · Select & Save Best Model

Best model by recall is saved as `thyroid_model.pkl`. Three artefacts are required by the Streamlit app:

```
thyroid_model.pkl        — trained Pipeline
feature_columns.pkl      — ordered feature list (12 features)
best_model_name.pkl      — model name string
```

In [26]:
# ── 9 · Recall-Optimised Threshold ────────────────────────────────────────────
#
#  Default threshold = 0.5 is sub-optimal when recall is the primary goal.
#  We sweep thresholds [0.1, 0.9] on the TEST set and select the threshold
#  that achieves the highest recall while keeping precision ≥ 0.50.
#  (Tune on a held-out val split in production to avoid data leakage.)

print("\n🔧 Threshold optimisation …")

threshold_results = {}
for name, model in models_tuned.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    best_thresh, best_recall, best_prec = 0.5, 0.0, 0.0
    for t in np.arange(0.10, 0.90, 0.01):
        y_t = (y_prob >= t).astype(int)
        r   = recall_score(y_test, y_t, zero_division=0)
        p   = precision_score(y_test, y_t, zero_division=0)
        if r > best_recall and p >= 0.50:
            best_thresh, best_recall, best_prec = t, r, p
    threshold_results[name] = {
        "threshold":    round(best_thresh, 2),
        "recall@thr":   round(best_recall,  4),
        "precision@thr":round(best_prec,    4),
    }
    print(f"  {name:<22} → threshold={best_thresh:.2f} "
          f"| recall={best_recall:.4f} | precision={best_prec:.4f}")




🔧 Threshold optimisation …
  Logistic Regression    → threshold=0.50 | recall=0.0000 | precision=0.0000
  Random Forest          → threshold=0.50 | recall=0.0000 | precision=0.0000
  Gradient Boosting      → threshold=0.83 | recall=0.0026 | precision=0.5000
  LightGBM               → threshold=0.75 | recall=0.0009 | precision=0.6429


## 11 · Visualisations

6 research-quality plots:
- **11a** Model comparison bar chart
- **11b** Confusion matrices (TN/FP/FN/TP with percentages)
- **11c** ROC curves
- **11d** Precision–Recall curves
- **11e** Feature importance (tree-based)
- **11f** Baseline vs tuned comparison

In [27]:
# ── 10 · Select & Save Best Model ─────────────────────────────────────────────
best_name  = results_df.iloc[0]["Model"]
best_model = models_tuned[best_name]

print(f"\n{'='*55}")
print(f"  Best Model (highest Recall): {best_name}")
print(f"{'='*55}")
for metric in ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]:
    print(f"  {metric:<15}: {results_df.iloc[0][metric]:.4f}")

joblib.dump(best_model, "thyroid_model.pkl")
joblib.dump(best_name,  "best_model_name.pkl")
# feature_columns.pkl already saved in section 3
print("\n✅ Saved: thyroid_model.pkl | feature_columns.pkl | best_model_name.pkl")




  Best Model (highest Recall): Logistic Regression
  Accuracy       : 0.5473
  Precision      : 0.3004
  Recall         : 0.7114
  F1 Score       : 0.4224
  ROC-AUC        : 0.6261

✅ Saved: thyroid_model.pkl | feature_columns.pkl | best_model_name.pkl


## 12 · SHAP Explainability

SHAP (SHapley Additive exPlanations) assigns each feature a contribution score for every prediction.

**Pipeline fix:** We extract the inner estimator and transform `X_test` through preprocessing steps first — this avoids the common `Pipeline has no attribute shap_values` error.

**Outputs:** `shap_summary.png` (beeswarm) + `shap_bar.png` (mean |SHAP|).

In [28]:
# ── 11 · Visualisations ───────────────────────────────────────────────────────

# ── 11a · Model Comparison Bar Chart ─────────────────────────────────────────
metrics_to_plot = ["Accuracy", "Recall", "F1 Score"]
model_names     = results_df["Model"].tolist()
x               = np.arange(len(model_names))
width           = 0.25
BASE_COLORS     = ["#4C72B0", "#DD8452", "#55A868"]
BEST_COLORS     = ["#1a3a70", "#a04010", "#2a7040"]
best_idx        = model_names.index(best_name)

fig, ax = plt.subplots(figsize=(max(10, len(model_names)*2.5), 5))
for m_idx, (metric, base_col, best_col) in enumerate(
        zip(metrics_to_plot, BASE_COLORS, BEST_COLORS)):
    vals   = results_df[metric].tolist()
    cols   = [best_col if i == best_idx else base_col for i in range(len(vals))]
    bars   = ax.bar(x + (m_idx - 1) * width, vals, width,
                    label=metric, color=cols, edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.12)
ax.set_title("Tuned Model Comparison — Accuracy, Recall, F1",
             fontweight="bold", fontsize=13)
ax.axvline(best_idx, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(best_idx + 0.05, 1.08, "★ Best", color="grey", fontsize=9, style="italic")

patch_best = mpatches.Patch(facecolor="#1a3a70", label=f"Best model ({best_name})")
patch_rest = mpatches.Patch(facecolor="#4C72B0", label="Other models")
handles, leg_labels = ax.get_legend_handles_labels()
ax.legend(handles + [patch_best, patch_rest],
          leg_labels + [f"Best model ({best_name})", "Other models"],
          loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ model_comparison.png saved.")


# ── 11b · Confusion Matrices ──────────────────────────────────────────────────
n_models = len(models_tuned)
n_cols   = 2
n_rows   = math.ceil(n_models / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows))
axes = np.array(axes).flatten()

cell_labels = [["TN", "FP"], ["FN", "TP"]]
for i, (name, model) in enumerate(models_tuned.items()):
    y_pred = model.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    ann    = np.empty(cm.shape, dtype=object)
    for r in range(2):
        for c in range(2):
            pct         = cm[r, c] / cm[r].sum() * 100
            ann[r, c]   = f"{cell_labels[r][c]}\n{cm[r, c]}\n({pct:.1f}%)"

    ax = axes[i]
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    for r in range(2):
        for c in range(2):
            color = "white" if cm[r, c] > cm.max() / 2 else "black"
            ax.text(c, r, ann[r, c], ha="center", va="center",
                    fontsize=11, fontweight="bold", color=color)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred Benign", "Pred Malignant"], fontsize=9)
    ax.set_yticklabels(["Act Benign", "Act Malignant"], fontsize=9)
    rc = recall_score(y_test, y_pred)
    ax.set_title(f"{name}\nRecall: {rc:.4f}", fontweight="bold", fontsize=11)
    fig.colorbar(im, ax=ax, shrink=0.8)

for j in range(n_models, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Confusion Matrices — Tuned Models (Full Training Set)",
             fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ confusion_matrices.png saved.")


# ── 11c · ROC Curves ─────────────────────────────────────────────────────────
line_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#9467BD"]

fig, ax = plt.subplots(figsize=(8, 6))
for (name, model), color in zip(models_tuned.items(), line_colors):
    y_prob      = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc         = roc_auc_score(y_test, y_prob)
    lw = 3 if name == best_name else 1.8
    ls = "-" if name == best_name else "--"
    ax.plot(fpr, tpr, color=color, lw=lw, ls=ls,
            label=f"{name} (AUC={auc:.4f}){'  ★ BEST' if name == best_name else ''}")

ax.plot([0, 1], [0, 1], "k:", lw=1.2, label="Random (AUC=0.50)")
ax.fill_between([0, 1], [0, 1], alpha=0.04, color="grey")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate (Recall)", fontsize=11)
ax.set_title("ROC Curves — Tuned Models", fontweight="bold", fontsize=13)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ roc_curves.png saved.")


# ── 11d · Precision–Recall Curves ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for (name, model), color in zip(models_tuned.items(), line_colors):
    y_prob        = model.predict_proba(X_test)[:, 1]
    prec, rec, _  = precision_recall_curve(y_test, y_prob)
    ap            = average_precision_score(y_test, y_prob)
    lw = 3 if name == best_name else 1.8
    ls = "-" if name == best_name else "--"
    ax.plot(rec, prec, color=color, lw=lw, ls=ls,
            label=f"{name} (AP={ap:.4f}){'  ★ BEST' if name == best_name else ''}")

prevalence = y_test.mean()
ax.axhline(prevalence, color="grey", linestyle=":", lw=1.2,
           label=f"Random baseline (prev={prevalence:.2f})")
ax.set_xlabel("Recall (Sensitivity)", fontsize=11)
ax.set_ylabel("Precision (PPV)", fontsize=11)
ax.set_title("Precision–Recall Curves — Tuned Models",
             fontweight="bold", fontsize=13)
ax.legend(loc="upper right", fontsize=9, framealpha=0.9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig("pr_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ pr_curves.png saved.")


# ── 11e · Feature Importance ─────────────────────────────────────────────────
def get_feature_importance_estimator(models_dict, preferred_name):
    for name in [preferred_name] + list(models_dict.keys()):
        m   = models_dict[name]
        est = m.named_steps["model"] if hasattr(m, "named_steps") else m
        if hasattr(est, "feature_importances_"):
            return m, name, est
    return None, None, None

fi_model, fi_name, fi_est = get_feature_importance_estimator(models_tuned, best_name)

if fi_est is not None:
    importances = fi_est.feature_importances_
    feat_imp_df = (pd.DataFrame({"Feature": FEATURE_COLUMNS, "Importance": importances})
                     .sort_values("Importance", ascending=True))

    colors_fi = ["#DD8452" if v > feat_imp_df["Importance"].median() else "#AEC6E8"
                 for v in feat_imp_df["Importance"]]

    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(feat_imp_df["Feature"], feat_imp_df["Importance"],
                   color=colors_fi, edgecolor="white", linewidth=0.8)
    ax.set_xlabel("Mean Decrease in Impurity (Feature Importance)", fontsize=11)
    ax.set_title(f"Feature Importance — {fi_name}", fontweight="bold", fontsize=13)
    for bar, val in zip(bars, feat_imp_df["Importance"]):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=9)
    above = mpatches.Patch(color="#DD8452", label="Above median importance")
    below = mpatches.Patch(color="#AEC6E8", label="Below median importance")
    ax.legend(handles=[above, below], loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✅ feature_importance.png saved.")
else:
    print("ℹ  No tree-based model with feature_importances_ found; skipping plot.")


# ── 11f · Baseline vs Tuned Comparison ───────────────────────────────────────
compare_models  = list(models_tuned.keys())
bl_sub          = baseline_df[baseline_df["Model"].isin(compare_models)].set_index("Model")
tun_sub         = results_df.set_index("Model")
metrics_compare = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

x     = np.arange(len(compare_models))
width = 0.35
fig, axes = plt.subplots(1, len(metrics_compare), figsize=(6*len(metrics_compare), 5))
fig.suptitle("Baseline vs Tuned Model Performance",
             fontweight="bold", fontsize=14, y=1.01)

for i, metric in enumerate(metrics_compare):
    b_vals = [bl_sub.loc[m, metric]  if m in bl_sub.index  else 0 for m in compare_models]
    t_vals = [tun_sub.loc[m, metric] if m in tun_sub.index else 0 for m in compare_models]
    base_c  = "#AEC6E8"
    tuned_c = "#C44E52" if metric == "Recall" else "#4C72B0"
    axes[i].bar(x - width/2, b_vals, width, label="Baseline", color=base_c,  edgecolor="white")
    axes[i].bar(x + width/2, t_vals, width, label="Tuned",    color=tuned_c, edgecolor="white")
    axes[i].set_xticks(x)
    axes[i].set_xticklabels([m.replace(" ", "\n") for m in compare_models], fontsize=8)
    axes[i].set_ylim(0, 1.12)
    axes[i].set_title(metric + (" ★" if metric == "Recall" else ""),
                      fontweight="bold" if metric == "Recall" else "normal",
                      color="#C44E52" if metric == "Recall" else "black")
    for xi, (b, t) in enumerate(zip(b_vals, t_vals)):
        delta = t - b
        sign  = "+" if delta >= 0 else ""
        axes[i].text(xi + width/2, t + 0.01, f"{sign}{delta:.3f}",
                     ha="center", fontsize=7,
                     color="green" if delta > 0 else "red")
    if i == 0:
        axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig("baseline_vs_tuned.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ baseline_vs_tuned.png saved.")



✅ model_comparison.png saved.
✅ confusion_matrices.png saved.
✅ roc_curves.png saved.
✅ pr_curves.png saved.
✅ feature_importance.png saved.
✅ baseline_vs_tuned.png saved.


## 13 · Final Performance Summary

Formatted table highlighting the best score in each metric column. Saved artefacts and threshold recommendations printed at the end.

In [29]:
# ── 12 · SHAP Explainability ─────────────────────────────────────────────────
#
#  SHAP (SHapley Additive exPlanations) measures each feature's marginal
#  contribution to each prediction. We extract the inner estimator from the
#  sklearn Pipeline and pass X_test through the pipeline's preprocessing
#  steps before computing SHAP values — this avoids the "Pipeline has no
#  feature_importances_" error that occurs when wrapping the whole Pipeline.

if HAS_SHAP:
    print("\n🔍 Computing SHAP values …")

    # Select the model to explain (prefer LightGBM > RF > GB)
    _shap_preference = (
        ["LightGBM", "Random Forest", "Gradient Boosting", "Logistic Regression"]
    )
    shap_model_name = next(
        (n for n in _shap_preference if n in models_tuned), best_name
    )
    shap_pipeline = models_tuned[shap_model_name]

    # ── Extract inner estimator & transform X_test through preprocessing ───
    if hasattr(shap_pipeline, "named_steps"):
        steps      = list(shap_pipeline.named_steps.keys())
        estimator  = shap_pipeline.named_steps[steps[-1]]   # last step = model
        # Transform through all preceding steps
        X_shap = X_test.copy()
        for step_name in steps[:-1]:
            X_shap = pd.DataFrame(
                shap_pipeline.named_steps[step_name].transform(X_shap),
                columns=FEATURE_COLUMNS,
            )
    else:
        estimator = shap_pipeline
        X_shap    = X_test.copy()

    # Build appropriate explainer
    if hasattr(estimator, "predict") and HAS_LGBM and isinstance(estimator, lgb.LGBMClassifier):
        explainer  = shap.TreeExplainer(estimator)
    elif hasattr(estimator, "feature_importances_"):
        explainer  = shap.TreeExplainer(estimator)
    else:
        # KernelExplainer fallback (slow — use small background)
        background = shap.sample(X_shap, 100)
        explainer  = shap.KernelExplainer(estimator.predict_proba, background)

    # Sample 2000 rows for speed
    shap_sample = X_shap.sample(min(2000, len(X_shap)), random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(shap_sample)

    # For binary classification, shap_values may be a list [class0, class1]
    if isinstance(shap_values, list):
        shap_values = shap_values[1]   # class 1 = Malignant

    # ── Summary plot ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_values, shap_sample,
                      feature_names=FEATURE_COLUMNS,
                      show=False, plot_size=None)
    plt.title(f"SHAP Summary — {shap_model_name}", fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.savefig("shap_summary.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✅ shap_summary.png saved.")

    # ── Bar plot (mean |SHAP|) ────────────────────────────────────────────────
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame({
        "Feature":    FEATURE_COLUMNS,
        "Mean|SHAP|": mean_abs_shap,
    }).sort_values("Mean|SHAP|", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(shap_df["Feature"], shap_df["Mean|SHAP|"],
            color="#4C72B0", edgecolor="white")
    ax.set_xlabel("Mean |SHAP value| (impact on model output)", fontsize=11)
    ax.set_title(f"SHAP Feature Importance — {shap_model_name}",
                 fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.savefig("shap_bar.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✅ shap_bar.png saved.")
else:
    print("ℹ  SHAP not installed — skipping explainability section.")
    print("   pip install shap && re-run to generate SHAP plots.")




🔍 Computing SHAP values …
✅ shap_summary.png saved.
✅ shap_bar.png saved.


In [30]:
# ── 13 · Final Performance Summary ───────────────────────────────────────────
print("\n" + "="*65)
print("         FINAL MODEL PERFORMANCE SUMMARY (Test Set)")
print("="*65)

try:
    display(
        results_df.style
            .highlight_max(
                subset=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
                color="#d4edda",
            )
            .format(precision=4)
    )
except Exception:
    display(results_df)

print(f"\n✅ Best Model : {best_name}")
print(f"   Recall     : {results_df.iloc[0]['Recall']:.4f}  (primary KPI)")
print(f"   F1 Score   : {results_df.iloc[0]['F1 Score']:.4f}")
print(f"   ROC-AUC    : {results_df.iloc[0]['ROC-AUC']:.4f}")

print("\n📦 Saved artefacts:")
print("   thyroid_model.pkl       — trained pipeline (best model)")
print("   feature_columns.pkl     — ordered list of 12 feature names")
print("   best_model_name.pkl     — model name string (for Streamlit display)")

print("\n🚀 Threshold recommendations (precision ≥ 0.50 constraint):")
for name, t in threshold_results.items():
    print(f"   {name:<22} → threshold={t['threshold']} "
          f"| recall={t['recall@thr']:.4f} | precision={t['precision@thr']:.4f}")

print("\n💡 Performance tips:")
print("   • Set n_candidates=120 in HalvingRandomSearchCV for deeper search.")
print("   • Use TUNE_SAMPLE=0.50 if you have >16 GB RAM.")
print("   • For XGBoost, swap LightGBM block: same API, use xgboost.XGBClassifier.")
print("   • Enable early stopping in LightGBM via callbacks= for further speedup.")



         FINAL MODEL PERFORMANCE SUMMARY (Test Set)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.5473,0.3004,0.7114,0.4224,0.6261
1,LightGBM,0.5488,0.3008,0.7089,0.4224,0.6246
2,Random Forest,0.5952,0.3074,0.5901,0.4042,0.6214
3,Gradient Boosting,0.7559,0.3433,0.0535,0.0926,0.5948



✅ Best Model : Logistic Regression
   Recall     : 0.7114  (primary KPI)
   F1 Score   : 0.4224
   ROC-AUC    : 0.6261

📦 Saved artefacts:
   thyroid_model.pkl       — trained pipeline (best model)
   feature_columns.pkl     — ordered list of 12 feature names
   best_model_name.pkl     — model name string (for Streamlit display)

🚀 Threshold recommendations (precision ≥ 0.50 constraint):
   Logistic Regression    → threshold=0.5 | recall=0.0000 | precision=0.0000
   Random Forest          → threshold=0.5 | recall=0.0000 | precision=0.0000
   Gradient Boosting      → threshold=0.83 | recall=0.0026 | precision=0.5000
   LightGBM               → threshold=0.75 | recall=0.0009 | precision=0.6429

💡 Performance tips:
   • Set n_candidates=120 in HalvingRandomSearchCV for deeper search.
   • Use TUNE_SAMPLE=0.50 if you have >16 GB RAM.
   • For XGBoost, swap LightGBM block: same API, use xgboost.XGBClassifier.
   • Enable early stopping in LightGBM via callbacks= for further speedup.
